In [1]:
# =========================
# Imports
# =========================

# pip install transformers datasets evaluate scikit-learn

import numpy as np
from datasets import load_dataset
from transformers import (
    set_seed,
    AutoConfig,
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)
import math
import torch
import torch.nn as nn
from collections import Counter
from google.colab import drive
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import DataLoader
import wandb

In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
os.environ['WANDB_API_KEY'] = user_secrets.get_secret("WANDB_API_KEY")

In [3]:
BASE_DIR = Path().absolute()
DATA_DIR = Path("/kaggle/input/competitions/spr-2026-mammography-report-classification").resolve()

In [4]:
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)
# print(torch.cuda.get_device_capability(0))

Tesla T4
12.8


In [5]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# =========================
# CONFIG
# =========================

# MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MODEL_NAME = "pucpr/biobertpt-all"
MAX_LENGTH = 512
BATCH_SIZE = 32
NUM_LABELS = 7

In [7]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [8]:
# =========================
# LOAD DATA
# =========================

dataset = load_dataset(
    "csv",
    data_files=str(DATA_DIR / 'train.csv')
)['train']

dataset = dataset.remove_columns("ID")
dataset = dataset.rename_column("report", "text")
dataset = dataset.rename_column("target", "labels")

dataset = dataset.shuffle(42)
dataset = dataset.class_encode_column('labels')
dataset = dataset.train_test_split(test_size=0.2, seed=2, stratify_by_column="labels")

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Stringifying the column:   0%|          | 0/18272 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/18272 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 14617
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 3655
    })
})

In [9]:
# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        # padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize_fn, batched=True, batch_size=BATCH_SIZE)
tokenized_dataset = tokenized_dataset.remove_columns(["token_type_ids", "text"])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/14617 [00:00<?, ? examples/s]

Map:   0%|          | 0/3655 [00:00<?, ? examples/s]

In [10]:
# # =========================
# # -------- STAGE 1 --------
# # DOMAIN ADAPTIVE PRETRAINING (MLM)
# # =========================

# config = AutoConfig.from_pretrained(MODEL_NAME)
# config.tie_word_embeddings = False

# mlm_model = AutoModelForMaskedLM.from_pretrained(
#     MODEL_NAME,
#     config=config
# )

# mlm_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=True,
#     mlm_probability=0.15
# )

# mlm_args = TrainingArguments(
#     output_dir="./mlm",
#     per_device_train_batch_size=BATCH_SIZE,
#     num_train_epochs=2,
#     learning_rate=5e-5,
#     warmup_steps=0.1,
#     lr_scheduler_type="cosine",
#     logging_steps=100,
#     eval_strategy='steps',
#     eval_steps=100,
#     save_strategy="no",
#     fp16=True,
#     report_to="none",
# )

# mlm_trainer = Trainer(
#     model=mlm_model,
#     args=mlm_args,
#     train_dataset=tokenized_dataset["train"],
#     eval_dataset=tokenized_dataset["test"],
#     data_collator=mlm_collator
# )

# mlm_trainer.train()

# # Save adapted encoder
# mlm_model.save_pretrained("adapted-bert")
# tokenizer.save_pretrained("adapted-bert")

In [11]:
# =========================
# -------- STAGE 2 --------
# SEQUENCE CLASSIFICATION
# =========================

# MODEL_NAME = str(BASE_DIR / "adapted-bert"),
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# Set dropout on the classifier head
model.config.classifier_dropout = 0.3
model.config._num_labels = 7

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: pucpr/biobertpt-all
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [12]:
# model.config

In [13]:
# =========================
# DATA COLLATOR
# =========================

cls_collator = DataCollatorWithPadding(tokenizer)

# =========================
# METRICS
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
    result = {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1-macro": float(f1_macro),
    }
    for i, f in enumerate(f1_per_class):
        result[f"f1_class_{i}"] = float(f)
    return result


# =========================
# SMOOTHED CLASS WEIGHTS
# =========================

labels = list(dataset['train']['labels'])
counts = Counter(labels)
total = sum(counts.values())

weights = []
for i in range(NUM_LABELS):
    cw = math.log1p(total / counts[i])
    weights.append(cw)

# alpha = 0.5  # tune this: 0.0 = raw inverse freq, 1.0 = very smooth (log1p)

# weights = []
# for i in range(NUM_LABELS):
#     freq = counts[i] / total
#     cw = freq ** (-alpha)   # e.g. alpha=0.5 gives sqrt(1/freq)
#     weights.append(cw)

mean_weight = sum(weights) / len(weights)
weights = [w / mean_weight for w in weights]

class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print("Original Counts:", dict(counts))
print("Smoothed Weights:", class_weights)

Original Counts: {2: 12774, 1: 555, 3: 570, 0: 488, 4: 171, 5: 23, 6: 36}
Smoothed Weights: tensor([0.8671, 0.8357, 0.1927, 0.8292, 1.1266, 1.6308, 1.5179],
       device='cuda:0')


In [14]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, logits, labels):
        # pt = true class probability (for focal modulation)
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)

        # CE with class weights, per-sample (no reduction)
        ce_loss = F.cross_entropy(
            logits, labels,
            weight=self.weight,
            reduction='none'
        )

        # Focal term: suppresses easy examples
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [15]:
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.model.training and self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = FocalLoss(weight=weights, gamma=self.gamma)
        else:
            loss_fct = nn.CrossEntropyLoss()    # plain CE for eval

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

In [16]:
wandb.init(
    project="spr-mammography",
    name="run6-focal-gamma2-pupcr",
    config={
        "model": MODEL_NAME,
        "class_weight_formula": "log1p",
        "focal_gamma": 2.0,
        "label_smoothing": None,       # removed for focal
        "classifier_dropout": 0.3,
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: lokeshtiwari001vns (lokeshtiwari001vns-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260331_031839-3y7ku9iz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run run6-focal-gamma2-pupcr
wandb: ⭐️ View project at https://wandb.ai/lokeshtiwari001vns-indian-institute-of-technology-madras/spr-mammography
wandb: 🚀 View run at https://wandb.ai/lokeshtiwari001vns-indian-institute-of-technology-madras/spr-mammography/runs/3y7ku9iz


In [17]:
# =========================
# TRAINING ARGS
# =========================

training_args = TrainingArguments(
    output_dir="./classifier",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=10,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_steps=0.15,
    weight_decay=0.05,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="f1-macro",
    fp16=True,
    report_to="wandb",
    save_total_limit=2,
)


# =========================
# TRAINER
# =========================

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=cls_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    class_weights=class_weights,
)

trainer.train()
wandb.finish()

Step,Training Loss,Validation Loss,Accuracy,F1-macro,F1 Class 0,F1 Class 1,F1 Class 2,F1 Class 3,F1 Class 4,F1 Class 5,F1 Class 6
500,0.064101,0.319083,0.886731,0.513535,0.734082,0.939502,0.945826,0.387097,0.588235,0.000000,0.000000
1000,0.026636,0.191461,0.940903,0.687421,0.738095,0.947368,0.976308,0.566845,0.666667,0.250000,0.666667
1500,0.012344,0.177506,0.949658,0.724169,0.791667,0.960289,0.980621,0.566154,0.659341,0.444444,0.666667
2000,0.005985,0.186374,0.949111,0.727153,0.782609,0.950000,0.979760,0.596386,0.681319,0.400000,0.700000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [18]:
# =========================
# SAVE FINAL MODEL
# =========================

trainer.save_model(BASE_DIR / "final-model")
tokenizer.save_pretrained(BASE_DIR / "final-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/final-model/tokenizer_config.json',
 '/kaggle/working/final-model/tokenizer.json')

In [19]:
import pandas as pd

train = pd.read_csv(DATA_DIR / "train.csv")
train.target.value_counts()

target
2    15968
3      713
1      693
0      610
4      214
6       45
5       29
Name: count, dtype: int64

In [20]:
# # =========================
# # INFERENCE TEST
# # =========================

# text = "Paciente apresenta sintomas de infecção respiratória."
# inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

# outputs = model(**inputs)
# pred = outputs.logits.argmax(dim=1).item()

# print("Prediction:", pred)